# Elo Merchant Category Recommendation — Raw EDA

Mục tiêu: đánh giá lịch sử giao dịch theo `card_id`. File historical khoảng 2.8GB,
vì vậy toàn bộ pipeline dùng Polars LazyFrame; chạy full EDA có thể mất nhiều phút.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import polars as pl

project_root = Path.cwd().resolve()
while project_root.parent != project_root and not (project_root / "src").exists():
    project_root = project_root.parent
if not (project_root / "src").exists():
    raise RuntimeError("Không tìm thấy project root chứa thư mục src")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda.config import DatasetSpec
from src.eda.descriptive import (
    describe_dataset,
    describe_entities,
    describe_quality,
    describe_temporal,
)
from src.eda.visualization import (
    plot_active_periods_per_entity,
    plot_entity_time_matrix,
    plot_missing_values,
    plot_records_over_time,
    plot_records_per_entity,
)

data_dir = project_root / "data" / "raw" / "elo-merchant-category-recommendation"
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

def scalar_metrics(report):
    return {key: value for key, value in report.items() if not isinstance(value, pl.DataFrame)}

## 1. Load và chuẩn hóa bảng sự kiện

In [3]:
historical = pl.scan_csv(
    data_dir / "historical_transactions.csv",
    try_parse_dates=True,
)
new_transactions = pl.scan_csv(
    data_dir / "new_merchant_transactions.csv",
    try_parse_dates=True,
)
transactions = pl.concat(
    [historical, new_transactions],
    how="diagonal_relaxed",
)
spec = DatasetSpec(
    name="elo_transactions",
    entity_col="card_id",
    time_col="purchase_date",
    categorical_cols=(
        "authorized_flag", "category_1", "category_2", "category_3",
        "city_id", "state_id", "merchant_category_id", "subsector_id",
    ),
    numeric_cols=("installments", "month_lag", "purchase_amount"),
    id_cols=("merchant_id",),
)

## 2. Dataset overview

In [4]:
dataset_report = describe_dataset(transactions, spec)
print("Shape:", dataset_report["n_rows"], "rows x", dataset_report["n_cols"], "columns")
print("Dtype groups:", dataset_report["dtype_groups"])
display(dataset_report["overview"].sort("missing_frac", descending=True))
display(dataset_report["preview"])

Shape: 31075392 rows x 14 columns
Dtype groups: {'numeric': ['city_id', 'installments', 'merchant_category_id', 'month_lag', 'purchase_amount', 'category_2', 'state_id', 'subsector_id'], 'string': ['authorized_flag', 'card_id', 'category_1', 'category_3', 'merchant_id'], 'categorical': [], 'temporal': ['purchase_date'], 'boolean': [], 'other': []}


column,dtype,n_missing,missing_frac,n_unique_approx
str,str,i64,f64,i64
"""category_2""","""Float64""",2764609,0.088965,6
"""category_3""","""String""",234081,0.007533,4
"""merchant_id""","""String""",164697,0.0053,322620
"""authorized_flag""","""String""",0,0.0,2
"""card_id""","""String""",0,0.0,364382
"""city_id""","""Int64""",0,0.0,308
"""category_1""","""String""",0,0.0,2
"""installments""","""Int64""",0,0.0,15
"""merchant_category_id""","""Int64""",0,0.0,321


authorized_flag,card_id,city_id,category_1,installments,category_3,merchant_category_id,merchant_id,month_lag,purchase_amount,purchase_date,category_2,state_id,subsector_id
str,str,i64,str,i64,str,i64,str,i64,f64,datetime[μs],f64,i64,i64
"""Y""","""C_ID_4e6213e9bc""",88,"""N""",0,"""A""",80,"""M_ID_e020e9b302""",-8,-0.703331,2017-06-25 15:33:07,1.0,16,37
"""Y""","""C_ID_4e6213e9bc""",88,"""N""",0,"""A""",367,"""M_ID_86ec983688""",-7,-0.733128,2017-07-15 12:10:45,1.0,16,16
"""Y""","""C_ID_4e6213e9bc""",88,"""N""",0,"""A""",80,"""M_ID_979ed661fc""",-6,-0.720386,2017-08-09 22:04:29,1.0,16,37
"""Y""","""C_ID_4e6213e9bc""",88,"""N""",0,"""A""",560,"""M_ID_e6d5ae8ea6""",-5,-0.735352,2017-09-02 10:06:26,1.0,16,34
"""Y""","""C_ID_4e6213e9bc""",88,"""N""",0,"""A""",80,"""M_ID_e020e9b302""",-11,-0.722865,2017-03-10 01:14:19,1.0,16,37


## 3. Data quality

In [ ]:
quality_report = describe_quality(transactions, spec)
print(scalar_metrics(quality_report))
display(quality_report["column_quality"].sort("missing_pct", descending=True))

## 4. Temporal coverage

In [ ]:
temporal_report = describe_temporal(transactions, spec)
print(scalar_metrics(temporal_report))
display(temporal_report["monthly_records"])

plot_records_over_time(temporal_report["monthly_records"])
plt.show()

## 5. Entity history

In [ ]:
entity_report = describe_entities(transactions, spec)
print(scalar_metrics(entity_report))
display(entity_report["entity_history"].head(20))

plot_records_per_entity(entity_report["entity_history"])
plt.show()
plot_active_periods_per_entity(entity_report["entity_history"])
plt.show()

## 6. Missing values

In [ ]:
plot_missing_values(quality_report["column_quality"], top_n=20)
plt.show()

## 7. Customer activity matrix

In [ ]:
matrix_result = plot_entity_time_matrix(
    transactions,
    entity_col=spec.entity_col,
    timestamp_col=spec.time_col,
    sample_size=12,
    min_active_periods=2,
    seed=42,
    id_suffix=10,
    annotate=True,
)
plt.show()
display(matrix_result.matrix)
display(matrix_result.entity_coverage)
print("Global coverage:", matrix_result.global_coverage)

## 8. Target và dataset-specific signals

In [ ]:
train = pl.scan_csv(data_dir / "train.csv", try_parse_dates=True)
target_summary = train.select(
    pl.len().alias("n_cards"),
    pl.col("target").null_count().alias("n_missing_target"),
    pl.col("target").mean().alias("mean"),
    pl.col("target").std().alias("std"),
    pl.col("target").median().alias("median"),
    pl.col("target").quantile(0.01).alias("p01"),
    pl.col("target").quantile(0.99).alias("p99"),
    pl.col("target").min().alias("min"),
    pl.col("target").max().alias("max"),
).collect()
display(target_summary)

authorized_distribution = (
    transactions.group_by("authorized_flag")
    .agg(pl.len().alias("n_transactions"))
    .sort("n_transactions", descending=True)
    .collect()
)
display(authorized_distribution)
merchants = pl.scan_csv(data_dir / "merchants.csv")
print("Merchants schema:", merchants.collect_schema())
display(merchants.head(5).collect())

## 9. Slide selection summary

Dataset có lịch sử card tốt và target sẵn, nhưng target là continuous loyalty
score chứ không phải binary propensity label. Chi phí compute cao nhất trong các bộ.

In [ ]:
slide_summary = pl.DataFrame(
    {
        "dataset": ["Elo"],
        "n_rows": [dataset_report["n_rows"]],
        "n_columns": [dataset_report["n_cols"]],
        "n_entities": [entity_report["n_unique_entities"]],
        "history_months": [temporal_report["history_months"]],
        "missing_months": [len(temporal_report["missing_months"])],
        "repeat_entity_pct": [entity_report["repeat_entity_pct"]],
        "active_2plus_months_pct": [entity_report["active_2plus_months_pct"]],
        "missing_cell_pct": [quality_report["missing_cell_pct"]],
        "duplicate_row_pct": [quality_report["duplicate_row_pct"]],
        "target_status": ["Có continuous loyalty target; không phải binary propensity"],
    }
)
display(slide_summary)